# Appendix C: Interactive DAG Exploration

**Causal Inference: A Pokemon Approach -- Kanto Region Edition**

---

This notebook lets you explore the full **Directed Acyclic Graph (DAG)** underlying the Kanto Trainer dataset. You'll be able to:

1. Visualise the complete causal structure
2. Highlight different path types (backdoor, frontdoor, mediated)
3. Identify valid adjustment sets for any treatment-outcome pair
4. See how `trainer_id` links datasets across the textbook

---

## 1. Setup

We try to use `graphviz` for publication-quality DAGs. If unavailable, we fall back to `networkx` + `matplotlib`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import combinations

from kanto_utils import apply_kanto_theme, oak_says, blue_says, nurse_joy_says
apply_kanto_theme()

# --- Rendering backend selection ---
USE_GRAPHVIZ = False
try:
    import graphviz
    USE_GRAPHVIZ = True
    print("Using graphviz for DAG rendering (publication quality).")
except ImportError:
    print("graphviz not available. Using networkx + matplotlib (still great!).")
    print("To install: pip install graphviz  (and install the system binary)")

import networkx as nx
print(f"networkx version: {nx.__version__}")

## 2. Building the Full Kanto Trainer DAG

The DAG below encodes the structural equations from `data/dgp/structural_equations.py`. Every arrow represents a direct causal effect in the data-generating process.

In [ ]:
# ===================================================================
# Define the DAG as a list of (parent, child) edges
# Grouped by structural equation for clarity
# ===================================================================

# --- Latent variables ---
LATENT_NODES = ['patience', 'natural_talent', 'dedication']

# --- Observable variables ---
OBSERVED_NODES = [
    'hometown', 'wealth', 'trainer_experience',
    'starter_type', 'team_size', 'team_level_avg', 'team_avg_iv_total',
    'type_diversity', 'play_hours', 'strategy_score',
    'cave_training', 'safari_zone_visits',
    'dept_store_spending', 'potions_purchased', 'tm_count',
    'exp_share_used', 'held_item_count', 'rare_candy_used',
    'badges', 'elite_four_attempted', 'elite_four_wins',
    'champion_defeated', 'total_battles_won', 'total_pokemon_caught',
]

# --- Edges (parent -> child) ---
EDGES = [
    # Hometown -> wealth
    ('hometown', 'wealth'),
    
    # Latent -> observable
    ('patience', 'safari_zone_visits'),
    ('patience', 'strategy_score'),
    ('patience', 'type_diversity'),
    ('natural_talent', 'strategy_score'),
    ('natural_talent', 'team_avg_iv_total'),
    ('dedication', 'play_hours'),
    ('dedication', 'potions_purchased'),
    ('dedication', 'tm_count'),
    ('dedication', 'cave_training'),
    ('dedication', 'team_level_avg'),
    
    # Wealth paths
    ('wealth', 'dept_store_spending'),
    ('wealth', 'starter_type'),
    ('wealth', 'exp_share_used'),
    ('wealth', 'rare_candy_used'),
    ('wealth', 'held_item_count'),
    
    # Experience paths
    ('trainer_experience', 'strategy_score'),
    ('trainer_experience', 'starter_type'),
    ('trainer_experience', 'cave_training'),
    ('trainer_experience', 'team_size'),
    ('trainer_experience', 'team_level_avg'),
    ('trainer_experience', 'type_diversity'),
    ('trainer_experience', 'tm_count'),
    ('trainer_experience', 'exp_share_used'),
    ('trainer_experience', 'play_hours'),
    
    # Play hours -> items / visits
    ('play_hours', 'potions_purchased'),
    ('play_hours', 'total_pokemon_caught'),
    
    # TM count -> held items
    ('tm_count', 'held_item_count'),
    
    # Multiple causes -> badges (the key outcome)
    ('strategy_score', 'badges'),
    ('team_level_avg', 'badges'),
    ('type_diversity', 'badges'),
    ('tm_count', 'badges'),
    ('dept_store_spending', 'badges'),
    ('exp_share_used', 'badges'),
    ('held_item_count', 'badges'),
    
    # Badges -> downstream outcomes
    ('badges', 'elite_four_attempted'),
    
    # Elite Four
    ('elite_four_attempted', 'elite_four_wins'),
    ('strategy_score', 'elite_four_wins'),
    ('team_level_avg', 'elite_four_wins'),
    ('type_diversity', 'elite_four_wins'),
    ('tm_count', 'elite_four_wins'),
    ('natural_talent', 'elite_four_wins'),
    ('elite_four_wins', 'champion_defeated'),
    
    # Battle wins
    ('trainer_experience', 'total_battles_won'),
    ('strategy_score', 'total_battles_won'),
    ('team_level_avg', 'total_battles_won'),
    
    # Pokemon caught
    ('safari_zone_visits', 'total_pokemon_caught'),
    ('patience', 'total_pokemon_caught'),
]

# Build the networkx DiGraph
G = nx.DiGraph()
G.add_nodes_from(LATENT_NODES + OBSERVED_NODES)
G.add_edges_from(EDGES)

print(f"DAG: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Is DAG: {nx.is_directed_acyclic_graph(G)}")

In [ ]:
oak_says(
    "A DAG (Directed Acyclic Graph) encodes our assumptions about <em>which</em> variables "
    "cause <em>which</em>. Every arrow means 'this variable has a direct causal effect on that one'. "
    "No arrow means 'no direct effect' -- though indirect effects can travel along paths. "
    "These assumptions determine which estimators will give you correct answers."
)

### 2.1 Visualise the Full DAG

In [ ]:
def draw_dag_networkx(G, latent_nodes, title="Kanto Trainer DAG",
                      highlight_edges=None, highlight_color='#EE1515',
                      figsize=(20, 14)):
    """Draw a DAG using networkx with a layered layout."""
    fig, ax = plt.subplots(figsize=figsize)
    
    # Use a topological layout with manual layer assignment
    # Group by causal order
    layers = {
        # Layer 0: Exogenous / latent
        'patience': 0, 'natural_talent': 0, 'dedication': 0,
        'hometown': 0, 'trainer_experience': 0,
        # Layer 1: First determined
        'wealth': 1, 'play_hours': 1,
        # Layer 2: Behaviours
        'starter_type': 2, 'strategy_score': 2, 'cave_training': 2,
        'safari_zone_visits': 2, 'team_avg_iv_total': 2,
        # Layer 3: Team + items
        'team_size': 3, 'team_level_avg': 3, 'type_diversity': 3,
        'dept_store_spending': 3, 'potions_purchased': 3,
        'tm_count': 3, 'rare_candy_used': 3,
        # Layer 4: Items derived
        'exp_share_used': 4, 'held_item_count': 4,
        # Layer 5: Core outcomes
        'badges': 5, 'total_battles_won': 5, 'total_pokemon_caught': 5,
        # Layer 6: Final outcomes
        'elite_four_attempted': 6, 'elite_four_wins': 6,
        'champion_defeated': 7,
    }
    
    # Assign positions: x = layer, y = spread within layer
    from collections import defaultdict
    layer_groups = defaultdict(list)
    for node, layer in layers.items():
        if node in G.nodes:
            layer_groups[layer].append(node)
    
    pos = {}
    for layer, nodes in layer_groups.items():
        n = len(nodes)
        for i, node in enumerate(sorted(nodes)):
            x = layer * 2.2
            y = (i - n / 2) * 1.5
            pos[node] = (x, y)
    
    # Assign any missing nodes
    for node in G.nodes:
        if node not in pos:
            pos[node] = (8, np.random.uniform(-5, 5))
    
    # Node colours
    node_colors = []
    for node in G.nodes:
        if node in latent_nodes:
            node_colors.append('#F95587')  # psychic pink for latent
        elif node == 'badges':
            node_colors.append('#FFD733')  # gold for badges
        elif node in ('elite_four_wins', 'champion_defeated'):
            node_colors.append('#EE8130')  # fire orange for final outcomes
        else:
            node_colors.append('#6390F0')  # water blue for observed
    
    # Edge colours
    edge_colors = []
    edge_widths = []
    if highlight_edges is None:
        highlight_edges = set()
    else:
        highlight_edges = set(highlight_edges)
    
    for edge in G.edges:
        if edge in highlight_edges:
            edge_colors.append(highlight_color)
            edge_widths.append(3.0)
        else:
            edge_colors.append('#AAAAAA')
            edge_widths.append(1.0)
    
    # Draw
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                           node_size=800, edgecolors='white', linewidths=1.5)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=edge_colors,
                           width=edge_widths, arrows=True,
                           arrowsize=15, arrowstyle='-|>',
                           connectionstyle='arc3,rad=0.1')
    
    # Labels with short names
    short_labels = {n: n.replace('_', '\n') for n in G.nodes}
    nx.draw_networkx_labels(G, pos, labels=short_labels, ax=ax,
                            font_size=7, font_weight='bold')
    
    # Legend
    legend_elements = [
        mpatches.Patch(facecolor='#F95587', edgecolor='white', label='Latent (unobserved)'),
        mpatches.Patch(facecolor='#6390F0', edgecolor='white', label='Observed'),
        mpatches.Patch(facecolor='#FFD733', edgecolor='white', label='Core outcome (badges)'),
        mpatches.Patch(facecolor='#EE8130', edgecolor='white', label='Final outcome'),
    ]
    if highlight_edges:
        legend_elements.append(
            mpatches.Patch(facecolor=highlight_color, edgecolor='white', label='Highlighted path')
        )
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10, frameon=True)
    
    ax.set_title(title, fontsize=18, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    return fig, ax


def draw_dag_graphviz(G, latent_nodes, title="Kanto Trainer DAG",
                      highlight_edges=None, highlight_color='red'):
    """Draw a DAG using graphviz for cleaner output."""
    dot = graphviz.Digraph(comment=title)
    dot.attr(rankdir='LR', size='16,10', fontsize='12')
    dot.attr('node', shape='box', style='rounded,filled', fontsize='10')
    
    if highlight_edges is None:
        highlight_edges = set()
    else:
        highlight_edges = set(highlight_edges)
    
    for node in G.nodes:
        if node in latent_nodes:
            dot.node(node, fillcolor='#F95587', fontcolor='white')
        elif node == 'badges':
            dot.node(node, fillcolor='#FFD733', fontcolor='#333')
        elif node in ('elite_four_wins', 'champion_defeated'):
            dot.node(node, fillcolor='#EE8130', fontcolor='white')
        else:
            dot.node(node, fillcolor='#6390F0', fontcolor='white')
    
    for u, v in G.edges:
        if (u, v) in highlight_edges:
            dot.edge(u, v, color=highlight_color, penwidth='3')
        else:
            dot.edge(u, v, color='#999999')
    
    return dot


# Draw the full DAG
if USE_GRAPHVIZ:
    dag_viz = draw_dag_graphviz(G, LATENT_NODES)
    display(dag_viz)
else:
    draw_dag_networkx(G, LATENT_NODES)
    plt.show()

## 3. Highlighting Path Types

Let's explore three fundamental path types in the DAG.

### 3.1 Backdoor Paths (Confounding)

A **backdoor path** from treatment to outcome passes through a common ancestor. If left open, it creates confounding bias.

**Example:** `exp_share_used -> badges` has a backdoor through `wealth`:
```
exp_share_used <-- wealth --> dept_store_spending --> badges
```

In [ ]:
# Highlight the backdoor path through wealth
backdoor_edges = [
    ('wealth', 'exp_share_used'),
    ('wealth', 'dept_store_spending'),
    ('dept_store_spending', 'badges'),
]

# Also highlight the direct causal path
direct_edges = [
    ('exp_share_used', 'badges'),
]

if USE_GRAPHVIZ:
    dag_bd = draw_dag_graphviz(G, LATENT_NODES, title="Backdoor Path Example",
                               highlight_edges=backdoor_edges + direct_edges,
                               highlight_color='red')
    display(dag_bd)
else:
    draw_dag_networkx(G, LATENT_NODES, title="Backdoor Path: wealth confounds exp_share -> badges",
                      highlight_edges=backdoor_edges + direct_edges,
                      highlight_color='#EE1515')
    plt.show()

print("RED EDGES: A backdoor path exists through 'wealth'.")
print("To estimate the causal effect of exp_share_used on badges,")
print("you must adjust for (at least) 'wealth' to close this path.")

### 3.2 Mediated Paths (Mechanism)

A **mediated path** shows *how* a cause operates. The effect flows through an intermediary.

**Example:** `dedication -> play_hours -> potions_purchased`

In [ ]:
mediated_edges = [
    ('dedication', 'play_hours'),
    ('play_hours', 'potions_purchased'),
    ('dedication', 'potions_purchased'),  # direct path also exists
]

if USE_GRAPHVIZ:
    dag_med = draw_dag_graphviz(G, LATENT_NODES, title="Mediated Path Example",
                                highlight_edges=mediated_edges,
                                highlight_color='#4DAD5B')
    display(dag_med)
else:
    draw_dag_networkx(G, LATENT_NODES,
                      title="Mediated Path: dedication -> play_hours -> potions_purchased",
                      highlight_edges=mediated_edges,
                      highlight_color='#4DAD5B')
    plt.show()

print("GREEN EDGES: Dedication affects potions both directly AND via play_hours.")
print("If you control for play_hours, you block the indirect path and only")
print("estimate the direct effect of dedication on potions.")

### 3.3 Collider Paths (Selection Bias)

A **collider** is a variable caused by two or more parents. Conditioning on a collider *opens* a previously blocked path, creating spurious associations.

**Example:** `strategy_score` is a collider for `patience` and `natural_talent`:
```
patience --> strategy_score <-- natural_talent
```
Conditioning on strategy_score induces an association between patience and natural_talent.

In [ ]:
collider_edges = [
    ('patience', 'strategy_score'),
    ('natural_talent', 'strategy_score'),
    ('trainer_experience', 'strategy_score'),
]

if USE_GRAPHVIZ:
    dag_col = draw_dag_graphviz(G, LATENT_NODES, title="Collider Example",
                                highlight_edges=collider_edges,
                                highlight_color='#FF7043')
    display(dag_col)
else:
    draw_dag_networkx(G, LATENT_NODES,
                      title="Collider: strategy_score is hit by patience, talent, experience",
                      highlight_edges=collider_edges,
                      highlight_color='#FF7043')
    plt.show()

blue_says(
    "I controlled for strategy_score to remove confounding, and now patience "
    "and natural_talent are correlated! My estimates are all messed up!"
)
oak_says(
    "That's because strategy_score is a <b>collider</b>. Conditioning on it "
    "<em>opens</em> the path between patience and natural_talent, creating "
    "a spurious association that wasn't there before. Never condition on "
    "a collider unless you have a very good reason!"
)

## 4. Interactive: Find Adjustment Sets for Any Treatment-Outcome Pair

Given a **treatment** and **outcome**, we can algorithmically identify:
1. All paths between them
2. Which paths are causal (directed from treatment to outcome)
3. Which paths are backdoor (go through a common ancestor)
4. Valid adjustment sets that block all backdoor paths without opening colliders

In [ ]:
def find_all_paths_undirected(G, source, target, max_length=8):
    """Find all simple paths in the underlying undirected graph."""
    G_undir = G.to_undirected()
    return list(nx.all_simple_paths(G_undir, source, target, cutoff=max_length))


def find_directed_paths(G, source, target, max_length=8):
    """Find all directed (causal) paths from source to target."""
    return list(nx.all_simple_paths(G, source, target, cutoff=max_length))


def find_backdoor_paths(G, treatment, outcome, max_length=8):
    """Find backdoor paths: paths from treatment to outcome that
    start with an arrow INTO treatment."""
    all_paths = find_all_paths_undirected(G, treatment, outcome, max_length)
    directed = find_directed_paths(G, treatment, outcome, max_length)
    directed_tuples = [tuple(p) for p in directed]
    
    backdoor = []
    for path in all_paths:
        if tuple(path) not in directed_tuples:
            backdoor.append(path)
    return backdoor


def get_parents(G, node):
    """Get all parents of a node in the DAG."""
    return set(G.predecessors(node))


def get_ancestors(G, node):
    """Get all ancestors of a node."""
    return nx.ancestors(G, node)


def suggest_adjustment_sets(G, treatment, outcome):
    """Suggest valid adjustment sets using the parent adjustment criterion.
    
    The simplest valid adjustment set is often the parents of the treatment
    (minus descendants of treatment on causal paths). This is a heuristic;
    for a complete algorithm, use the full backdoor criterion.
    """
    # Parents of treatment (excluding outcome)
    parents_of_treatment = get_parents(G, treatment) - {outcome}
    
    # Descendants of treatment (should NOT be adjusted for)
    descendants = nx.descendants(G, treatment)
    
    # Method 1: Adjust for parents of treatment
    set1 = parents_of_treatment - descendants
    
    # Method 2: Adjust for all common causes of treatment and outcome
    ancestors_t = get_ancestors(G, treatment)
    ancestors_o = get_ancestors(G, outcome)
    common_ancestors = ancestors_t & ancestors_o
    set2 = common_ancestors - descendants - {treatment, outcome}
    
    return {
        'parents_of_treatment': set1,
        'common_ancestors': set2,
    }


print("Path-finding functions defined. Ready for interactive exploration!")

In [ ]:
# =============================================
# INTERACTIVE: Change these variables to explore!
# =============================================

TREATMENT = 'exp_share_used'
OUTCOME = 'badges'

# --- Analysis ---
print(f"Treatment: {TREATMENT}")
print(f"Outcome:   {OUTCOME}")
print("=" * 60)

# Directed (causal) paths
causal_paths = find_directed_paths(G, TREATMENT, OUTCOME)
print(f"\nCausal paths ({len(causal_paths)} found):")
for i, path in enumerate(causal_paths, 1):
    print(f"  {i}. {' -> '.join(path)}")

# Backdoor paths
backdoor = find_backdoor_paths(G, TREATMENT, OUTCOME)
print(f"\nBackdoor paths ({len(backdoor)} found):")
for i, path in enumerate(backdoor, 1):
    print(f"  {i}. {' -- '.join(path)}")

# Adjustment sets
adj_sets = suggest_adjustment_sets(G, TREATMENT, OUTCOME)
print(f"\nSuggested adjustment sets:")
print(f"  Parents of treatment:  {adj_sets['parents_of_treatment'] or '{empty -- no confounders!}'}")
print(f"  Common ancestors:      {adj_sets['common_ancestors'] or '{empty}'}")

# Descendants (should NOT adjust for)
desc = nx.descendants(G, TREATMENT)
print(f"\nDescendants of treatment (DO NOT adjust for these):")
print(f"  {desc & set(OBSERVED_NODES) if desc else '{none}'}")

In [ ]:
# Visualise the paths for this treatment-outcome pair
all_highlighted = set()
for path in causal_paths:
    for i in range(len(path) - 1):
        all_highlighted.add((path[i], path[i + 1]))

# Add backdoor edges in both directions (they may go either way)
for path in backdoor:
    for i in range(len(path) - 1):
        if G.has_edge(path[i], path[i + 1]):
            all_highlighted.add((path[i], path[i + 1]))
        elif G.has_edge(path[i + 1], path[i]):
            all_highlighted.add((path[i + 1], path[i]))

if USE_GRAPHVIZ:
    dag_paths = draw_dag_graphviz(G, LATENT_NODES,
                                  title=f"Paths: {TREATMENT} -> {OUTCOME}",
                                  highlight_edges=all_highlighted,
                                  highlight_color='red')
    display(dag_paths)
else:
    draw_dag_networkx(G, LATENT_NODES,
                      title=f"All paths: {TREATMENT} -> {OUTCOME}",
                      highlight_edges=all_highlighted,
                      highlight_color='#EE1515')
    plt.show()

In [ ]:
nurse_joy_says(
    "Try changing <code>TREATMENT</code> and <code>OUTCOME</code> in the cell above "
    "to explore different causal questions! Some interesting pairs to try:<br><br>"
    "- <code>safari_zone_visits</code> -> <code>total_pokemon_caught</code> (confounded by patience)<br>"
    "- <code>cave_training</code> -> <code>badges</code> (indirect via team_level)<br>"
    "- <code>wealth</code> -> <code>champion_defeated</code> (long causal chain)<br>"
    "- <code>starter_type</code> -> <code>badges</code> (is starter choice causal?)"
)

## 5. Cross-Dataset Linkage via `trainer_id`

One of the unique features of this textbook is that many datasets share a `trainer_id` key, allowing you to link information across different causal contexts.

In [ ]:
from kanto_utils import (load_trainers, load_battles, load_protein_rct,
                          load_ss_anne, load_safari_lottery)

# Load all datasets that share trainer_id
datasets = {
    'kanto_trainers': load_trainers(),
    'kanto_battles': load_battles(),
    'pewter_protein_rct': load_protein_rct(),
    'ss_anne_passengers': load_ss_anne(),
    'safari_zone_lottery': load_safari_lottery(),
}

print("Datasets with trainer_id:\n")
for name, df in datasets.items():
    has_tid = 'trainer_id' in df.columns
    n_unique = df['trainer_id'].nunique() if has_tid else 0
    print(f"  {name:<25s}  trainer_id: {'Yes':>3s}  "
          f"unique IDs: {n_unique:>6,}  rows: {len(df):>6,}")

In [ ]:
# Visualise the linkage structure
fig, ax = plt.subplots(figsize=(14, 7))

# Create a linkage diagram
link_G = nx.DiGraph()

# Add dataset nodes
ds_nodes = [
    ('kanto_trainers', '2,000 trainers\n42 columns'),
    ('kanto_battles', '50,000 battles\n28 columns'),
    ('pewter_protein_rct', '200 trainers\n12 columns'),
    ('ss_anne_passengers', '400 passengers\n15 columns'),
    ('safari_zone_lottery', '600 trainers\n13 columns'),
    ('kanto_cities_panel', '192 city-months\n12 columns'),
    ('happiness_evolution', '800 pokemon\n13 columns'),
]

for name, desc in ds_nodes:
    link_G.add_node(name, label=f"{name}\n({desc})")

# Add linkage edges
link_G.add_edge('kanto_trainers', 'kanto_battles', label='trainer_id\n(1:many)')
link_G.add_edge('kanto_trainers', 'pewter_protein_rct', label='trainer_id\n(subset)')
link_G.add_edge('kanto_trainers', 'ss_anne_passengers', label='trainer_id\n(subset)')
link_G.add_edge('kanto_trainers', 'safari_zone_lottery', label='trainer_id\n(subset)')

# Layout
pos = {
    'kanto_trainers': (0, 0),
    'kanto_battles': (3, 2),
    'pewter_protein_rct': (3, 0.7),
    'ss_anne_passengers': (3, -0.7),
    'safari_zone_lottery': (3, -2),
    'kanto_cities_panel': (-3, 1.5),
    'happiness_evolution': (-3, -1.5),
}

# Draw nodes
linked = ['kanto_trainers', 'kanto_battles', 'pewter_protein_rct',
          'ss_anne_passengers', 'safari_zone_lottery']
standalone = ['kanto_cities_panel', 'happiness_evolution']

nx.draw_networkx_nodes(link_G, pos, nodelist=linked, ax=ax,
                       node_color='#3B4CCA', node_size=3000,
                       node_shape='s', edgecolors='white', linewidths=2)
nx.draw_networkx_nodes(link_G, pos, nodelist=standalone, ax=ax,
                       node_color='#4DAD5B', node_size=3000,
                       node_shape='s', edgecolors='white', linewidths=2)

# Draw edges
nx.draw_networkx_edges(link_G, pos, ax=ax, edge_color='#EE1515',
                       width=2.5, arrows=True, arrowsize=20,
                       arrowstyle='-|>')

# Labels
short_labels = {n: n.replace('_', '\n') for n in link_G.nodes}
nx.draw_networkx_labels(link_G, pos, labels=short_labels, ax=ax,
                        font_size=8, font_weight='bold', font_color='white')

# Edge labels
edge_labels = nx.get_edge_attributes(link_G, 'label')
nx.draw_networkx_edge_labels(link_G, pos, edge_labels=edge_labels, ax=ax,
                              font_size=8, font_color='#EE1515')

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#3B4CCA', edgecolor='white', label='Linked by trainer_id'),
    mpatches.Patch(facecolor='#4DAD5B', edgecolor='white', label='Standalone datasets'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=11, frameon=True)

ax.set_title('Cross-Dataset Linkage Map\n(trainer_id connects the Kanto universe)',
             fontsize=16, fontweight='bold')
ax.axis('off')
fig.tight_layout()
plt.show()

In [ ]:
# Demonstrate a cross-dataset merge
trainers = load_trainers()
battles = load_battles()

# How many battles does each trainer have?
battle_counts = battles.groupby('trainer_id').size().reset_index(name='n_battles')
merged = trainers.merge(battle_counts, on='trainer_id', how='left')
merged['n_battles'] = merged['n_battles'].fillna(0).astype(int)

print(f"Merged dataset: {merged.shape}")
print(f"\nCorrelation between play_hours and number of battles in the battles dataset:")
print(f"  r = {merged['play_hours'].corr(merged['n_battles']):.3f}")
print(f"\nThis makes sense: the DGP assigns battles proportional to play_hours!")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(merged['play_hours'], merged['n_battles'], alpha=0.2, s=10, color='#3B4CCA')
ax.set_xlabel('Play Hours (from kanto_trainers)')
ax.set_ylabel('Number of Battles (from kanto_battles)')
ax.set_title('Cross-Dataset Linkage: Trainers x Battles')
plt.tight_layout()
plt.show()

In [ ]:
oak_says(
    "The ability to link datasets is a powerful feature of this textbook. "
    "In the real world, you often have information scattered across multiple "
    "data sources. Merging on <code>trainer_id</code> lets you combine "
    "RCT data with observational covariates, battle outcomes with trainer "
    "characteristics, and more. Just be careful: merging can introduce "
    "new colliders if you condition on post-treatment variables!"
)

## 6. Quick Reference: Node Roles in the DAG

The table below classifies every variable by its role in the causal structure.

In [ ]:
# Generate a summary table of node roles
rows = []
for node in sorted(G.nodes):
    parents = list(G.predecessors(node))
    children = list(G.successors(node))
    role = 'Latent' if node in LATENT_NODES else 'Observed'
    if len(parents) == 0:
        causal_role = 'Root (exogenous)'
    elif len(children) == 0:
        causal_role = 'Terminal (outcome only)'
    elif len(parents) >= 2:
        causal_role = 'Collider (multiple parents)'
    else:
        causal_role = 'Mediator'
    
    rows.append({
        'Variable': node,
        'Status': role,
        'Causal Role': causal_role,
        'Parents': len(parents),
        'Children': len(children),
    })

role_df = pd.DataFrame(rows).sort_values(['Status', 'Causal Role', 'Variable'])
role_df.set_index('Variable', inplace=True)

# Style the table
def highlight_role(row):
    if row['Status'] == 'Latent':
        return ['background-color: #FFF0F4'] * len(row)
    elif 'Collider' in row['Causal Role']:
        return ['background-color: #FFF3EB'] * len(row)
    elif 'Root' in row['Causal Role']:
        return ['background-color: #F0F9F0'] * len(row)
    return [''] * len(row)

role_df.style.apply(highlight_role, axis=1)